# Sillon : entraîner le diagnostic photo

Ce notebook entraîne le petit modèle (environ 1 Mo) qui reconnaît sur le téléphone, sans Internet :
**feuille saine, mildiou, oïdium, alternariose**, et « **autre problème** » pour le reste.

Avant de lancer (« Run All ») :
1. **Settings > Accelerator : GPU T4** (gratuit, environ 30 h par semaine).
2. **Settings > Internet : On** (pour récupérer le script et les poids ImageNet).
3. **Add Data** : ajoutez un jeu **PlantVillage** (photos de labo) et un jeu **PlantDoc** (photos de terrain).
   Le notebook les retrouve tout seul dans `/kaggle/input`.

Durée : 1 à 2 heures. À la fin, téléchargez `sillon_disease.tflite` et `sillon_disease.json`
(onglet Output) et placez-les dans `app/src/main/assets/models/` du dépôt Sillon.

In [ ]:
# Le script d'entraînement, depuis le dépôt Sillon.
!git clone --depth 1 https://github.com/Relthdrarxash/sillon.git /kaggle/working/sillon
%cd /kaggle/working/sillon/tools/ml

In [ ]:
# Repère automatiquement les jeux ajoutés avec « Add Data ».
from pathlib import Path
inputs = [p for p in Path('/kaggle/input').iterdir() if p.is_dir()]
plantvillage = [p for p in inputs if 'village' in p.name.lower()]
plantdoc = [p for p in inputs if 'doc' in p.name.lower()]
print('PlantVillage :', plantvillage)
print('PlantDoc     :', plantdoc)
assert plantdoc, "Ajoutez un jeu PlantDoc : sans photos de terrain, le modèle ne vaut rien au jardin."
args = ' '.join(f'--plantvillage "{p}"' for p in plantvillage) + ' ' + ' '.join(f'--plantdoc "{p}"' for p in plantdoc)

In [ ]:
!python train_sillon.py {args} --out /kaggle/working/out

## Lire le résultat

Dans `sillon_disease.json`, regardez **`metrics.field_test`** : ce sont les photos de terrain jamais vues.
- `macro_f1` : qualité moyenne par classe. En dessous de 0,60, ne publiez pas le modèle.
- `threshold` : seuil de confiance retenu pour que les réponses affichées soient justes au moins 85 % du temps.
- `field_coverage_at_threshold` : part des photos pour lesquelles l'appli osera répondre (sinon, elle renvoie vers la clé de diagnostic).
- `confusion` : lignes = vraie classe, colonnes = classe prédite, dans l'ordre sain, mildiou, oïdium, alternariose, autre.

In [ ]:
import json
meta = json.load(open('/kaggle/working/out/sillon_disease.json'))
field = meta['metrics']['field_test']
print('F1 terrain :', field['macro_f1'], ' précision :', field['accuracy'], ' ECE :', field['ece'])
print('Seuil :', meta['threshold'], ' couverture :', meta['metrics']['field_coverage_at_threshold'])
for name, m in field['per_class'].items():
    print(f"  {name:13s} F1 {m['f1']:.2f}  (n = {m['n']})")